# 12 - Molecular Subtype Baseline Comparison

This notebook addresses the reviewer concern (Dr. Jack Virostko, UT Austin) that our top predictors (Ki-67, age, ER status) are already known to correlate with molecular subtype, and asks: **"Is your analysis providing better prediction than this sub-typing?"**

## Why This Analysis Matters

PAM50 molecular subtypes (Luminal A, Luminal B, HER2-enriched, Basal-like, Normal-like) are the standard clinical classification for breast cancer. If our combined model were merely learning to distinguish subtypes, a classifier using only subtype labels should match its performance. We test this directly.

## Analysis Overview

1. **Subtype-only baseline**: One-hot encode PAM50 subtype → 5 features → same 3 classifiers + Cox model under 5-fold CV
2. **Comparison**: Combined model AUC/C-index vs. subtype-only AUC/C-index
3. **Within-subtype stratification**: Can we separate high- vs. low-risk patients *within* Luminal A (n=719)?

## What Are PAM50 Subtypes?

The PAM50 classifier assigns breast tumors to one of five intrinsic subtypes based on expression of 50 genes:
- **Luminal A**: ER+, low Ki-67, best prognosis
- **Luminal B**: ER+, higher Ki-67, more aggressive than Luminal A
- **HER2-enriched**: HER2 overexpression, aggressive without targeted therapy
- **Basal-like**: Triple-negative, high proliferation, worst prognosis
- **Normal-like**: Gene expression resembling normal breast tissue

**Expected results:**
- Subtype-only AUC: 0.613 (identical across all 3 classifiers)
- Combined model AUC: 0.856 (RF), delta = +0.243
- Subtype-only Cox C-index: 0.613; Combined: 0.827, delta = +0.214
- Within Luminal A: CV C-index 0.848 +/- 0.047, OOF log-rank p = 5.86e-22

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Prepare Data

We load the GSE96058 clinical data and encode features identically to the main pipeline.

In [ ]:
# Load clinical data
clin = pd.read_csv('../data/clinical/01_gse96058_clinical.csv')
df = clin.copy()

# Encode clinical features (same as original pipeline)
cat_cols = ['lymph_node_status', 'er_status', 'pgr_status', 'her2_status',
            'ki67_status', 'nhg', 'pam50_subtype']
for c in cat_cols:
    df[c] = df[c].fillna('Unknown').astype(str)
    le = LabelEncoder()
    df[c + '_enc'] = le.fit_transform(df[c])

num_cols = ['age_at_diagnosis', 'tumor_size']
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

y = df['high_risk'].astype(int).values

print(f"Total patients: {len(df)}")
print(f"High risk: {y.sum()} ({y.mean():.1%})")
print(f"\nPAM50 subtype distribution:")
print(df['pam50_subtype'].value_counts().to_string())

## 2. Subtype-Only Binary Classification

One-hot encode PAM50 subtype to create 5 binary features, then run same 3 classifiers under 5-fold stratified CV.

In [ ]:
# One-hot encode PAM50 subtype
subtype_dummies = pd.get_dummies(df['pam50_subtype'], prefix='sub')
X_subtype = subtype_dummies.values

print(f"Subtype features ({X_subtype.shape[1]}): {list(subtype_dummies.columns)}")

# Define classifiers (same hyperparameters as main pipeline)
classifiers = {
    'Elastic Net': lambda: LogisticRegression(penalty='elasticnet', l1_ratio=0.5, C=1.0,
                                               solver='saga', max_iter=5000,
                                               class_weight='balanced', random_state=42),
    'Random Forest': lambda: RandomForestClassifier(n_estimators=500, max_depth=8,
                                                     class_weight='balanced',
                                                     random_state=42, n_jobs=-1),
    'Gradient Boosting': lambda: GradientBoostingClassifier(n_estimators=500,
                                                             learning_rate=0.03,
                                                             max_depth=3, random_state=42)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results_subtype = {}
for name, clf_fn in classifiers.items():
    aucs = []
    for train_idx, test_idx in skf.split(X_subtype, y):
        X_tr, X_te = X_subtype[train_idx], X_subtype[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)
        clf = clf_fn()
        clf.fit(X_tr_s, y_tr)
        y_prob = clf.predict_proba(X_te_s)[:, 1]
        aucs.append(roc_auc_score(y_te, y_prob))
    m, s = np.mean(aucs), np.std(aucs)
    results_subtype[name] = (m, s)
    print(f"{name}: AUC = {m:.3f} +/- {s:.3f}")

print("\nNote: All classifiers yield identical AUC because 5 one-hot features")
print("have limited capacity — all converge to the same effective decision rule.")

## 3. Subtype-Only Cox C-index

Evaluate Cox PH using only subtype indicators as features.

In [ ]:
# Prepare survival data with subtype features
surv_df = subtype_dummies.copy()
surv_df['time'] = df['time_to_event'].values
surv_df['event'] = df['event_status'].values
surv_df = surv_df[surv_df['time'] > 0].copy()
feat_sub = [c for c in surv_df.columns if c.startswith('sub_')]
event_bin = (surv_df['event'] > 0).astype(int).values

cox_cis = []
for train_idx, test_idx in skf.split(surv_df[feat_sub], event_bin):
    train_d = surv_df.iloc[train_idx].copy()
    test_d = surv_df.iloc[test_idx].copy()
    sc = StandardScaler()
    train_d[feat_sub] = sc.fit_transform(train_d[feat_sub])
    test_d[feat_sub] = sc.transform(test_d[feat_sub])
    try:
        cph = CoxPHFitter(penalizer=0.5)
        cph.fit(train_d, duration_col='time', event_col='event')
        pred = cph.predict_partial_hazard(test_d)
        ci = concordance_index(test_d['time'], -pred.values.flatten(), test_d['event'])
        cox_cis.append(ci)
    except Exception as e:
        print(f"Fold error: {e}")

print(f"Subtype-Only Cox C-index: {np.mean(cox_cis):.3f} +/- {np.std(cox_cis):.3f}")

## 4. Comparison: Subtype-Only vs. Combined Model

Print the full comparison table with previously computed combined model results.

In [ ]:
# Combined model results (from notebooks 04 and 06)
combined_results = {
    'Elastic Net': 0.855,
    'Random Forest': 0.856,
    'Gradient Boosting': 0.827,
}
combined_cox = 0.827

print("=" * 65)
print("SUBTYPE-ONLY vs. COMBINED MODEL COMPARISON")
print("=" * 65)
print(f"{'Model':<20} {'Subtype Only':>12} {'Combined':>10} {'Delta':>8}")
print("-" * 65)

rows = []
for name in classifiers:
    sub_auc = results_subtype[name][0]
    sub_std = results_subtype[name][1]
    comb_auc = combined_results[name]
    delta = comb_auc - sub_auc
    print(f"{name:<20} {sub_auc:>12.3f} {comb_auc:>10.3f} {delta:>+8.3f}")
    rows.append({
        'model': name,
        'subtype_only_auc': round(sub_auc, 3),
        'subtype_only_std': round(results_subtype[name][1], 3),
        'combined_auc': comb_auc,
        'delta': f"+{delta:.3f}"
    })

# Cox C-index
sub_cox = np.mean(cox_cis)
delta_cox = combined_cox - sub_cox
print(f"{'Cox C-index':<20} {sub_cox:>12.3f} {combined_cox:>10.3f} {delta_cox:>+8.3f}")
rows.append({
    'model': 'Cox C-index',
    'subtype_only_auc': round(sub_cox, 3),
    'subtype_only_std': round(np.std(cox_cis), 3),
    'combined_auc': combined_cox,
    'delta': f"+{delta_cox:.3f}"
})
print("=" * 65)

# Save results
results_df = pd.DataFrame(rows)
results_df.to_csv('../results/subtype_baseline_results.csv', index=False)
print("\nSaved to results/subtype_baseline_results.csv")

## 5. Within-Subtype Risk Stratification: Luminal A

This is the strongest evidence that our model adds value beyond subtype classification.

**Why within-subtype stratification matters:** If the model can separate high-risk from low-risk patients *within a single subtype*, it proves that the clinical features capture prognostic variation that the subtype label alone cannot resolve. We focus on Luminal A because:
1. It is the largest subtype group (n=719), providing adequate statistical power
2. It is the "best prognosis" subtype — showing risk heterogeneity here is clinically impactful
3. Luminal A patients are sometimes considered uniformly low-risk, which this analysis challenges

**Critical methodological note:** We use out-of-fold (OOF) predictions to avoid data leakage. Each patient's risk score comes from a model that never saw that patient during training.

In [ ]:
# Filter to Luminal A patients
luma = df[df['pam50_subtype'] == 'LumA'].copy()

# Clinical features EXCLUDING pam50_subtype_enc (since all are LumA)
clin_no_pam = ['lymph_node_status_enc', 'er_status_enc', 'pgr_status_enc',
               'her2_status_enc', 'ki67_status_enc', 'nhg_enc',
               'age_at_diagnosis', 'tumor_size']

surv_luma = luma[clin_no_pam].copy().astype(float)
surv_luma['time'] = luma['time_to_event'].values
surv_luma['event'] = luma['event_status'].values
surv_luma = surv_luma[surv_luma['time'] > 0].copy()
event_bin = (surv_luma['event'] > 0).astype(int).values

print(f"Luminal A patients: {len(surv_luma)}")
print(f"Events: {event_bin.sum()} ({event_bin.mean():.1%})")
print(f"Features (excluding PAM50): {clin_no_pam}")

# Collect out-of-fold predictions for honest evaluation
oof_preds = np.zeros(len(surv_luma))
cv_cis = []
for train_idx, test_idx in skf.split(surv_luma[clin_no_pam], event_bin):
    train_d = surv_luma.iloc[train_idx].copy()
    test_d = surv_luma.iloc[test_idx].copy()
    sc = StandardScaler()
    train_d[clin_no_pam] = sc.fit_transform(train_d[clin_no_pam])
    test_d[clin_no_pam] = sc.transform(test_d[clin_no_pam])
    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(train_d, duration_col='time', event_col='event')
    pred = cph.predict_partial_hazard(test_d)
    oof_preds[test_idx] = pred.values.flatten()
    ci = concordance_index(test_d['time'], -pred.values.flatten(), test_d['event'])
    cv_cis.append(ci)

t = surv_luma['time'].values
e = surv_luma['event'].values
risk_groups = (oof_preds > np.median(oof_preds)).astype(int)

lr = logrank_test(t[risk_groups==0], t[risk_groups==1],
                  e[risk_groups==0], e[risk_groups==1])

print(f"\nLuminal A Within-Subtype Results:")
print(f"  n = {len(surv_luma)}")
print(f"  Events = {event_bin.sum()}")
print(f"  CV C-index = {np.mean(cv_cis):.3f} +/- {np.std(cv_cis):.3f}")
print(f"  OOF log-rank p = {lr.p_value:.2e}")
print(f"  High-risk group: n={risk_groups.sum()}, events={(e[risk_groups==1]>0).sum()}")
print(f"  Low-risk group: n={(risk_groups==0).sum()}, events={(e[risk_groups==0]>0).sum()}")

## 6. Kaplan-Meier Figure: Luminal A Within-Subtype

Generate the KM figure using out-of-fold risk assignments (no data leakage).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
kmf = KaplanMeierFitter()

mask_low = risk_groups == 0
kmf.fit(t[mask_low], e[mask_low], label=f'Low Risk (n={mask_low.sum()})')
kmf.plot_survival_function(ax=ax, color='#2196F3', linewidth=2.5, ci_show=True, ci_alpha=0.12)

mask_high = risk_groups == 1
kmf.fit(t[mask_high], e[mask_high], label=f'High Risk (n={mask_high.sum()})')
kmf.plot_survival_function(ax=ax, color='#F44336', linewidth=2.5, ci_show=True, ci_alpha=0.12)

ax.set_xlabel('Time (months)', fontsize=13)
ax.set_ylabel('Survival Probability', fontsize=13)
ax.set_title('Within-Subtype Risk Stratification: Luminal A Patients', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='lower left')

pstr = f'Log-rank p = {lr.p_value:.2e}'
ax.text(0.95, 0.95, pstr, transform=ax.transAxes, fontsize=11,
        ha='right', va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.8))
ax.text(0.95, 0.87, f'CV C-index = {np.mean(cv_cis):.3f}', transform=ax.transAxes, fontsize=10,
        ha='right', va='top')

ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/fig_km_luminal_a.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved to figures/fig_km_luminal_a.png")

## 7. Save Within-Subtype Results

In [ ]:
within_df = pd.DataFrame([{
    'subtype': 'LumA',
    'n_patients': len(surv_luma),
    'n_events': int(event_bin.sum()),
    'cv_c_index': round(np.mean(cv_cis), 3),
    'cv_c_index_std': round(np.std(cv_cis), 3),
    'oof_logrank_p': f"{lr.p_value:.2e}"
}])
within_df.to_csv('../results/within_subtype_results.csv', index=False)
print("Saved to results/within_subtype_results.csv")
print(within_df.to_string(index=False))

## 8. Summary: All Key Numbers for Paper Verification

Print all numbers that appear in the IEEE JBHI paper for reviewer verification.

In [ ]:
print("=" * 70)
print("MOLECULAR SUBTYPE BASELINE — COMPLETE RESULTS SUMMARY")
print("=" * 70)

print("\n--- Subtype-Only Binary Classification (5-fold CV) ---")
for name, (m, s) in results_subtype.items():
    print(f"  {name}: AUC = {m:.3f} +/- {s:.3f}")

print(f"\n--- Subtype-Only Cox C-index ---")
print(f"  C-index = {np.mean(cox_cis):.3f} +/- {np.std(cox_cis):.3f}")

print(f"\n--- Combined vs. Subtype-Only Deltas ---")
for name in classifiers:
    delta = combined_results[name] - results_subtype[name][0]
    print(f"  {name}: +{delta:.3f}")
print(f"  Cox C-index: +{combined_cox - np.mean(cox_cis):.3f}")

print(f"\n--- Within-Subtype: Luminal A ---")
print(f"  n = {len(surv_luma)}")
print(f"  Events = {int(event_bin.sum())}")
print(f"  CV C-index = {np.mean(cv_cis):.3f} +/- {np.std(cv_cis):.3f}")
print(f"  OOF log-rank p = {lr.p_value:.2e}")

print("\n" + "=" * 70)
print("All numbers above should match the IEEE JBHI paper (Tables IV, V).")
print("=" * 70)